Code chính

In [2]:
import pandas as pd
from backtesting import Backtest, Strategy
import math
from vnstock3 import Vnstock
import talib as ta
import seaborn as sns
import matplotlib.pyplot as plt

RSI_PERIOD = 14
RSI_OVERSOLD = 30
MACD_FAST = 12
MACD_SLOW = 26
MACD_SIGNAL = 9

CODE CHÍNH

# vnd

In [7]:
def calculate_first_mondays(dates):
    if not isinstance(dates, pd.DatetimeIndex):
        dates = pd.DatetimeIndex(dates)
    dates_series = pd.Series(dates, index=dates)
    mondays = dates_series[dates_series.dt.dayofweek == 0]
    first_mondays = mondays.groupby([mondays.dt.year, mondays.dt.month]).first()
    return set(first_mondays)
class DCA(Strategy):
    average_monthly_income_vnd = 1000  # Average monthly income in VND
    investment_percentage = 0.10  # Percentage of income to invest
    fund = 0 # Initialize the investment fund 

    def init(self):
        close = self.data.Close
        # Calculate RSI
        self.rsi = self.I(ta.RSI, close, timeperiod=RSI_PERIOD) 
        # Calculate MACD
        macd, signal_line, _ = ta.MACD(close, fastperiod=MACD_FAST, slowperiod=MACD_SLOW, signalperiod=MACD_SIGNAL)  
        self.macd = self.I(pd.Series, macd)
        self.signal_line = self.I(pd.Series, signal_line)
        self.previous_macd = self.I(pd.Series(macd).shift, 1)
        self.previous_signal_line = self.I(pd.Series(signal_line).shift, 1)
        self.first_mondays = calculate_first_mondays(self.data.index)

    def next(self):
        # Update the fund at the start of each month
        today = self.data.index[-1]
        self.data.Close[-1] = self.data.Close[-1] / 10
        if today in self.first_mondays:
            self.fund += self.average_monthly_income_vnd * self.investment_percentage
            
        # Check for buy signal: RSI cross above 30 and MACD cross above Signal line
        if (self.previous_macd[-1] < self.previous_signal_line[-1] and
            self.macd[-1] >= self.signal_line[-1] and
            self.rsi[-1] > RSI_OVERSOLD):  
            share_price = self.data.Close[-1]
            max_shares_to_buy = self.fund // share_price
            lots_to_buy = max_shares_to_buy // 100
            
            if lots_to_buy > 0:
                shares_to_buy = lots_to_buy * 100
                self.buy(size=shares_to_buy)
                self.fund -= share_price * shares_to_buy
                #print(f"Buy executed at {self.data.index[-1]} with {shares_to_buy} shares at price {share_price}, total price {share_price * shares_to_buy}")



def run_backtest(stock_symbol):
    # Fetch stock data
    stock_data = Vnstock().stock(symbol=stock_symbol,source='VCI').quote.history(start='2019-01-01', end='2024-01-04')
    stock_data = stock_data.rename(columns={"open": "Open", "high": "High", "low": "Low", "close": "Close", "volume": "Volume"})
    stock_data.set_index('time', inplace=True)
    stock_data.index = pd.to_datetime(stock_data.index)
    stock_data.index = stock_data.index.normalize()
    stock_data = stock_data.dropna()

    # Run the backtest
    bt = Backtest(
        stock_data,
        DCA,
        trade_on_close=True,
    )
    stats = bt.run()
    #bt.plot(filename=f'{stock_symbol}')
    
    # Calculate investment details
    trades = stats["_trades"]
    price_paid = trades["Size"] * trades["EntryPrice"]
    total_invested = price_paid.sum()

    current_shares = trades["Size"].sum()
    current_equity = current_shares * stock_data.Close.iloc[-1]
    print(trades)
    print(f"Results for {stock_symbol}:")
    print("Total investment:", total_invested)
    print("Current Shares:", current_shares)
    print("Current Equity:", current_equity)
    print("RoR:", ((current_equity - total_invested) / total_invested)*100)
    print("-" * 50)

# List of stock symbols
stock_symbols = ['FPT', 'MWG', 'E1VFVN30']

# Run backtest for each stock
for symbol in stock_symbols:
    run_backtest(symbol)

2024-10-26 17:45:27,580 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS
2024-10-26 17:45:28,322 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


    Size  EntryBar  ExitBar  EntryPrice  ExitPrice    PnL  ReturnPct  \
0    100      1194     1251       8.299      8.316    1.7   0.002048   
1    100      1056     1251       5.811      8.316  250.5   0.431079   
2    100       886     1251       6.165      8.316  215.1   0.348905   
3    100       771     1251       5.361      8.316  295.5   0.551203   
4    100       675     1251       5.672      8.316  264.4   0.466150   
5    100       525     1251       3.698      8.316  461.8   1.248783   
6    100       447     1251       2.714      8.316  560.2   2.064112   
7    100       397     1251       2.337      8.316  597.9   2.558408   
8    100       376     1251       2.382      8.316  593.4   2.491184   
9    100       311     1251       2.025      8.316  629.1   3.106667   
10   100       275     1251       2.315      8.316  600.1   2.592225   
11   100       236     1251       2.430      8.316  588.6   2.422222   
12   100       146     1251       2.081      8.316  623.5   2.99

2024-10-26 17:45:28,799 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


    Size  EntryBar  ExitBar  EntryPrice  ExitPrice    PnL  ReturnPct  \
0    100      1213     1251       3.998      4.286   28.8   0.072036   
1    100      1126     1251       4.471      4.286  -18.5  -0.041378   
2    100      1048     1251       3.872      4.286   41.4   0.106921   
3    100       970     1251       4.142      4.286   14.4   0.034766   
4    100       887     1251       6.338      4.286 -205.2  -0.323761   
5    100       751     1251       6.634      4.286 -234.8  -0.353934   
6    100       592     1251       4.866      4.286  -58.0  -0.119194   
7    100       494     1251       3.890      4.286   39.6   0.101799   
8    100       397     1251       2.537      4.286  174.9   0.689397   
9    100       354     1251       2.883      4.286  140.3   0.486646   
10   100       311     1251       2.236      4.286  205.0   0.916816   
11   100       279     1251       3.535      4.286   75.1   0.212447   
12   100       173     1251       3.951      4.286   33.5   0.08